# Step 5: Clean PPD Plan-Level Data

Replicates `code/clean_ppd.do` — the final stage of the pipeline.

- **Input:** `ppd_plan_clean_allocations.dta` (Step 3)
- **Merges:** `general_consultants_final.dta`, `zipcodes.csv`, `ppd_performance.dta` (Step 4)
- **Output:** `ppd_plan_level_clean.dta` (4000 rows × 291 cols)
- **Baseline:** `output/ppd_plan_level_clean.dta`

In [1]:
import pandas as pd
import numpy as np
import pyreadstat
from pathlib import Path

ROOT = Path.cwd().parent
RAW = ROOT / 'raw'
CODE = ROOT / 'legacy' / 'code'
OUTPUT_STATA = ROOT / 'legacy' / 'output'
OUTPUT_PY = ROOT / 'output_python'

def load_dta(path):
    df, meta = pyreadstat.read_dta(str(path))
    return df

print(f'Root: {ROOT}')

Root: /Users/Work/Desktop/Pension Research/ppd-data


## 1. Load Reference Tables

In [2]:
# Zipcodes (Stata: import delimited → tempfile)
zipcodes = pd.read_csv(RAW / 'zipcodes.csv')
print(f'Zipcodes: {zipcodes.shape}, cols={list(zipcodes.columns)}')
print(f'  ppd_id dups: {zipcodes.duplicated(subset=["ppd_id"]).sum()}')

# Consultants
consultants = load_dta(RAW / 'general_consultants_final.dta')
print(f'Consultants: {consultants.shape}')
print(f'  (pub_id, fy) dups: {consultants.duplicated(subset=["pub_id","fy"]).sum()}')

# Performance (Step 4 Python output)
performance = load_dta(OUTPUT_PY / 'ppd_performance.dta')
print(f'Performance: {performance.shape}')
print(f'  (pub_id, fy) dups: {performance.duplicated(subset=["pub_id","fy"]).sum()}')

Zipcodes: (229, 3), cols=['ppd_id', 'planname', 'ppd_zipcode']
  ppd_id dups: 0
Consultants: (4074, 9)
  (pub_id, fy) dups: 0
Performance: (3780, 9)
  (pub_id, fy) dups: 0


## 2. Load Step 3 Data & Apply Filters

In [3]:
# Stata: use ppd_plan_clean_allocations.dta, clear
df = load_dta(OUTPUT_PY / 'ppd_plan_clean_allocations.dta')
n0 = len(df)
print(f'Loaded Step 3: {df.shape}')

# Stata: drop if mi(pub_id)
df = df[df['pub_id'].notna() & (df['pub_id'] != '')].copy()
n1 = len(df)
print(f'After drop missing pub_id: {n1} (dropped {n0 - n1})')

# Stata: keep if inrange(fy, 2001, 2021)
df = df[df['fy'].between(2001, 2021)].copy()
n2 = len(df)
print(f'After keep fy 2001-2021: {n2} (dropped {n1 - n2})')

# Stata: keep if ~mi(MktAssets_net) & ~mi(ActLiabilities_GASB)
df = df[df['MktAssets_net'].notna() & df['ActLiabilities_GASB'].notna()].copy()
n3 = len(df)
print(f'After keep non-missing assets/liabilities: {n3} (dropped {n2 - n3})')

Loaded Step 3: (6268, 284)
After drop missing pub_id: 5016 (dropped 1252)
After keep fy 2001-2021: 4788 (dropped 228)
After keep non-missing assets/liabilities: 4635 (dropped 153)


## 3. Allocation Weight Validation & Replacement

In [4]:
WEIGHT_THRESH = 0.05

PPD_ASSETS = ['EQTotal', 'FITotal', 'RETotal', 'AltMiscTotal', 'PETotal',
              'HFTotal', 'COMDTotal', 'CashTotal', 'OtherTotal']

# Stata: replace X_Actl = 0 if mi(X_Actl); ppd_actl_sum += X_Actl (for each asset)
# This fills NaN → 0 in-place for each asset, then sums
for v in PPD_ASSETS:
    df[f'{v}_Actl'] = df[f'{v}_Actl'].fillna(0)
    df[f'{v}_Trgt'] = df[f'{v}_Trgt'].fillna(0)

actl_cols = [f'{v}_Actl' for v in PPD_ASSETS]
trgt_cols = [f'{v}_Trgt' for v in PPD_ASSETS]

df['ppd_actl_sum'] = df[actl_cols].sum(axis=1)
df['ppd_trgt_sum'] = df[trgt_cols].sum(axis=1)

# Stata: gen ppd_actl_check = abs(ppd_actl_sum - 1) < weight_thresh
df['ppd_actl_check'] = (np.abs(df['ppd_actl_sum'] - 1) < WEIGHT_THRESH).astype(float)
df['ppd_trgt_check'] = (np.abs(df['ppd_trgt_sum'] - 1) < WEIGHT_THRESH).astype(float)

# Stata: drop if ~ppd_actl_check & ~ppd_trgt_check
n_before = len(df)
df = df[~((df['ppd_actl_check'] == 0) & (df['ppd_trgt_check'] == 0))].copy()
print(f'After weight drop: {len(df)} (dropped {n_before - len(df)})')

# Stata: gen is_actual_replaced / is_target_replaced
df['is_actual_replaced'] = ((df['ppd_trgt_check'] == 1) & (df['ppd_actl_check'] == 0)).astype(float)
df['is_target_replaced'] = ((df['ppd_actl_check'] == 1) & (df['ppd_trgt_check'] == 0)).astype(float)

print(f'Rows with actual replaced by target: {int(df["is_actual_replaced"].sum())}')
print(f'Rows with target replaced by actual: {int(df["is_target_replaced"].sum())}')

# Stata: foreach v { replace v_Actl = v_Trgt if is_actual_replaced; 
#                     replace v_Trgt = v_Actl if is_target_replaced }
# CRITICAL: target replacement uses the CURRENT (potentially already-replaced) v_Actl
# But in the Stata loop, is_actual_replaced and is_target_replaced are MUTUALLY EXCLUSIVE
# (a row can't have both actl_check=0 AND actl_check=1), so order doesn't matter.
for v in PPD_ASSETS:
    actl_mask = df['is_actual_replaced'] == 1
    trgt_mask = df['is_target_replaced'] == 1
    df.loc[actl_mask, f'{v}_Actl'] = df.loc[actl_mask, f'{v}_Trgt']
    df.loc[trgt_mask, f'{v}_Trgt'] = df.loc[trgt_mask, f'{v}_Actl']

print(f'Weight validation complete. Shape: {df.shape}')

After weight drop: 4000 (dropped 635)
Rows with actual replaced by target: 80
Rows with target replaced by actual: 406
Weight validation complete. Shape: (4000, 290)


## 4. Membership Cleanup

In [5]:
# Stata: egen member_check = rowtotal(beneficiaries_tot actives_tot
#     InactiveVestedMembers InactiveNonVested)
member_cols = ['beneficiaries_tot', 'actives_tot', 'InactiveVestedMembers', 'InactiveNonVested']
df['member_check'] = df[member_cols].fillna(0).sum(axis=1)

# Stata: replace TotMembership = member_check if pub_id == "CO1004" || mi(TotMembership)
cpera_or_missing = (df['pub_id'] == 'CO1004') | df['TotMembership'].isna()
df.loc[cpera_or_missing, 'TotMembership'] = df.loc[cpera_or_missing, 'member_check']

# Stata: replace TotMembership = . if TotMembership == 0
df.loc[df['TotMembership'] == 0, 'TotMembership'] = np.nan

# Stata: replace beneficiaries_tot = . if beneficiaries_tot == 0
df.loc[df['beneficiaries_tot'] == 0, 'beneficiaries_tot'] = np.nan

print(f'TotMembership nulls: {df["TotMembership"].isna().sum()}')
print(f'beneficiaries_tot nulls: {df["beneficiaries_tot"].isna().sum()}')

TotMembership nulls: 15
beneficiaries_tot nulls: 25


## 5. Merge Consultant, Zipcodes, Performance

In [6]:
n_pre = len(df)

# ── Merge 1: Consultants (m:1 pub_id fy, keep 1 3, keepusing general_consultant consultant_id) ──
cons_keep = consultants[['pub_id', 'fy', 'general_consultant', 'consultant_id']].copy()
df = df.merge(cons_keep, on=['pub_id', 'fy'], how='left')
print(f'After consultant merge: {len(df)} rows (was {n_pre})')
print(f'  general_consultant non-null: {df["general_consultant"].notna().sum()}')

# ── Merge 2: Zipcodes (m:1 ppd_id, keep 1 3) ──
df = df.merge(zipcodes, on='ppd_id', how='left')
print(f'After zipcode merge: {len(df)} rows')
print(f'  ppd_zipcode non-null: {df["ppd_zipcode"].notna().sum()}')

# ── Merge 3: Performance (m:1 pub_id fy, keep 1 3, keepusing ret_bgnassets) ──
perf_keep = performance[['pub_id', 'fy', 'ret_bgnassets']].copy()
df = df.merge(perf_keep, on=['pub_id', 'fy'], how='left')
print(f'After performance merge: {len(df)} rows')
print(f'  ret_bgnassets non-null: {df["ret_bgnassets"].notna().sum()}')

assert len(df) == n_pre, f'Row count changed after merges: {n_pre} -> {len(df)}'

After consultant merge: 4000 rows (was 4000)
  general_consultant non-null: 4000
After zipcode merge: 4000 rows
  ppd_zipcode non-null: 4000
After performance merge: 4000 rows
  ret_bgnassets non-null: 3998


## 6. Drop Temp Variables & Sort

In [7]:
# Stata: drop member_check ppd_actl_sum ppd_trgt_sum ppd_actl_check ppd_trgt_check
drop_cols = ['member_check', 'ppd_actl_sum', 'ppd_trgt_sum', 'ppd_actl_check', 'ppd_trgt_check']
df = df.drop(columns=drop_cols)

# Stata: xtset ppd_id fy (implies sort by ppd_id fy)
df = df.sort_values(['ppd_id', 'fy']).reset_index(drop=True)

print(f'Final shape: {df.shape}')
print(f'Unique (ppd_id, fy): {df.duplicated(subset=["ppd_id","fy"]).sum()} duplicates')

Final shape: (4000, 291)
Unique (ppd_id, fy): 0 duplicates


## 7. Save Outputs

In [8]:
out_name = 'ppd_plan_level_clean'

pyreadstat.write_dta(df, str(OUTPUT_PY / f'{out_name}.dta'))
df.to_parquet(OUTPUT_PY / f'{out_name}.parquet', index=False)
df.to_csv(OUTPUT_PY / f'{out_name}.csv', index=False)

print(f'Saved: {out_name}.dta, .parquet, .csv')
print(f'Shape: {df.shape}')

Saved: ppd_plan_level_clean.dta, .parquet, .csv
Shape: (4000, 291)


## 8. Structural Parity Check

In [9]:
stata_df = load_dta(OUTPUT_STATA / 'ppd_plan_level_clean.dta')
py_df = load_dta(OUTPUT_PY / 'ppd_plan_level_clean.dta')

results = []

# Shape
shape_ok = stata_df.shape == py_df.shape
results.append(('Shape', shape_ok, f'Stata={stata_df.shape} Py={py_df.shape}'))

# Column set
col_set_ok = set(stata_df.columns) == set(py_df.columns)
only_st = sorted(set(stata_df.columns) - set(py_df.columns))
only_py = sorted(set(py_df.columns) - set(stata_df.columns))
results.append(('Column set', col_set_ok, f'Only Stata={only_st} Only Py={only_py}'))

# Column order
col_order_ok = list(stata_df.columns) == list(py_df.columns)
results.append(('Column order', col_order_ok, ''))

# Key uniqueness
st_dups = stata_df.duplicated(subset=['ppd_id', 'fy']).sum()
py_dups = py_df.duplicated(subset=['ppd_id', 'fy']).sum()
results.append(('Key uniqueness', st_dups == 0 and py_dups == 0, f'St dups={st_dups} Py dups={py_dups}'))

for name, ok, detail in results:
    status = 'PASS' if ok else 'FAIL'
    extra = f' — {detail}' if (not ok and detail) else ''
    print(f'{name}: {status}{extra}')

Shape: PASS
Column set: PASS
Column order: PASS
Key uniqueness: PASS


## 9. Deep Value Parity

In [10]:
keys = ['ppd_id', 'fy']
common = sorted(set(py_df.columns) & set(stata_df.columns))

py_s = py_df[common].sort_values(keys).reset_index(drop=True)
st_s = stata_df[common].sort_values(keys).reset_index(drop=True)

# Normalize strings
for c in common:
    if pd.api.types.is_string_dtype(st_s[c]):
        st_s[c] = st_s[c].str.strip().replace('', np.nan)
    if pd.api.types.is_string_dtype(py_s[c]):
        py_s[c] = py_s[c].str.strip().replace('', np.nan)

mismatches = {}
for col in common:
    p = py_s[col]
    s = st_s[col]
    nan_diff = int((p.isna() != s.isna()).sum())
    both_valid = p.notna() & s.notna()
    if both_valid.any():
        if pd.api.types.is_numeric_dtype(p) or pd.api.types.is_numeric_dtype(s):
            pv = pd.to_numeric(p[both_valid], errors='coerce').astype(np.float64)
            sv = pd.to_numeric(s[both_valid], errors='coerce').astype(np.float64)
            val_diff = int((~np.isclose(pv, sv, rtol=1e-6, atol=1e-8, equal_nan=True)).sum())
        else:
            val_diff = int((p[both_valid].astype(str) != s[both_valid].astype(str)).sum())
    else:
        val_diff = 0
    total = nan_diff + val_diff
    if total > 0:
        mismatches[col] = {'nan': nan_diff, 'val': val_diff}

if mismatches:
    print(f'MISMATCHES in {len(mismatches)} column(s):')
    for c, info in sorted(mismatches.items()):
        print(f'  {c}: nan_diff={info["nan"]}, val_diff={info["val"]}')
else:
    print(f'All {len(common)} columns match across {len(py_s)} rows.')

print(f'\n{"EXACT PARITY CONFIRMED" if len(mismatches) == 0 else "PARITY ISSUES DETECTED"}')

MISMATCHES in 7 column(s):
  CashTotal_Trgt: nan_diff=0, val_diff=19
  EQTotal_Trgt: nan_diff=0, val_diff=24
  FITotal_Trgt: nan_diff=0, val_diff=24
  HFTotal_Trgt: nan_diff=0, val_diff=1
  PETotal_Trgt: nan_diff=0, val_diff=24
  RETotal_Trgt: nan_diff=0, val_diff=24
  trgt_zero_actl_nonzero: nan_diff=0, val_diff=24

PARITY ISSUES DETECTED
